=============================================================================
PIPELINE CLEANING & PENGGABUNGAN DATASET
Sistem Rekomendasi Karir Mahasiswa IT Berbasis Machine Learning
=============================================================================
 
Dataset yang digunakan:
1. train-00000-of-00001.parquet  → Job Postings IT (dari HuggingFace batuhanmtl/job-skill-set)
2. survey_results_public.csv     → StackOverflow Developer Survey 2024
 
Output:
- dataset_it_careers_clean.csv   → Dataset gabungan siap untuk training model ML
 
Jalankan:
    pip install pandas numpy scikit-learn pyarrow imbalanced-learn matplotlib seaborn
    python career_dataset_pipeline.py

In [1]:
import pandas as pd
import numpy as np
import ast
import re
import warnings
import os
warnings.filterwarnings('ignore')
 


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found

In [2]:
# KONFIGURASI PATH
PATH_PARQUET = "train-00000-of-00001.parquet"
PATH_CSV     = "survey_results_public.csv"
PATH_OUTPUT  = "dataset_it_careers_clean.csv"

In [3]:
# MAPPING: Label Karir Target (35 Karir IT)
# Mapping dari keyword pada job_title (Dataset 1)
TITLE_TO_CAREER = {
    # Software Development
    "frontend developer":        "Frontend Developer",
    "front-end developer":       "Frontend Developer",
    "front end developer":       "Frontend Developer",
    "ui developer":              "Frontend Developer",
    "backend developer":         "Backend Developer",
    "back-end developer":        "Backend Developer",
    "back end developer":        "Backend Developer",
    "full stack":                "Full Stack Developer",
    "fullstack":                 "Full Stack Developer",
    "mobile developer":          "Mobile App Developer",
    "mobile app":                "Mobile App Developer",
    "android developer":         "Mobile App Developer",
    "ios developer":             "Mobile App Developer",
    "game developer":            "Game Developer",
    "game dev":                  "Game Developer",
    "software engineer":         "Backend Developer",
    "software developer":        "Backend Developer",
    "application developer":     "Full Stack Developer",
    # Data & AI
    "data analyst":              "Data Analyst",
    "business analyst":          "Business Intelligence Analyst",
    "business intelligence":     "Business Intelligence Analyst",
    "bi analyst":                "Business Intelligence Analyst",
    "data engineer":             "Data Engineer",
    "data scientist":            "Data Scientist",
    "machine learning":          "Machine Learning Engineer",
    "ml engineer":               "Machine Learning Engineer",
    "ai engineer":               "Machine Learning Engineer",
    "data science":              "Data Scientist",
    # Infrastruktur & Cloud
    "system administrator":      "System Administrator",
    "sysadmin":                  "System Administrator",
    "network engineer":          "Network Engineer",
    "network administrator":     "Network Engineer",
    "network admin":             "Network Engineer",
    "cloud engineer":            "Cloud Engineer",
    "cloud architect":           "Cloud Engineer",
    "cloud infrastructure":      "Cloud Engineer",
    "devops":                    "DevOps Engineer",
    "site reliability":          "Site Reliability Engineer (SRE)",
    "sre":                       "Site Reliability Engineer (SRE)",
    "infrastructure engineer":   "DevOps Engineer",
    # Cybersecurity
    "security analyst":          "Cybersecurity Analyst",
    "cybersecurity":             "Cybersecurity Analyst",
    "cyber security":            "Cybersecurity Analyst",
    "information security":      "Cybersecurity Analyst",
    "penetration tester":        "Penetration Tester",
    "pen tester":                "Penetration Tester",
    "security engineer":         "Security Engineer",
    "digital forensics":         "Digital Forensics Analyst",
    "forensic":                  "Digital Forensics Analyst",
    # Database
    "database administrator":    "Database Administrator (DBA)",
    "dba":                       "Database Administrator (DBA)",
    "database developer":        "Database Developer",
    "database engineer":         "Database Administrator (DBA)",
    # UI/UX & Product
    "ui designer":               "UI Designer",
    "ui/ux":                     "UX Designer",
    "ux designer":               "UX Designer",
    "ux researcher":             "UX Researcher",
    "product manager":           "Product Manager (Teknologi)",
    # QA & Testing
    "qa engineer":               "QA Engineer",
    "quality assurance":         "QA Engineer",
    "test engineer":             "Automation Test Engineer",
    "automation test":           "Automation Test Engineer",
    # IT Support
    "it support":                "IT Support Specialist",
    "helpdesk":                  "Helpdesk Analyst",
    "help desk":                 "Helpdesk Analyst",
    "technical support":         "Technical Support Engineer",
    "support specialist":        "IT Support Specialist",
    "support technician":        "IT Support Specialist",
    "support analyst":           "IT Support Specialist",
}

In [4]:
# Mapping dari DevType StackOverflow ke label karir target
SO_DEVTYPE_TO_CAREER = {
    "Developer, front-end":                    "Frontend Developer",
    "Developer, back-end":                     "Backend Developer",
    "Developer, full-stack":                   "Full Stack Developer",
    "Developer, mobile":                       "Mobile App Developer",
    "Developer, game or graphics":             "Game Developer",
    "Developer, desktop or enterprise applications": "Backend Developer",
    "Developer, embedded applications or devices":   "Backend Developer",
    "Developer, AI apps or physical AI":       "Machine Learning Engineer",
    "Data or business analyst":                "Data Analyst",
    "Data engineer":                           "Data Engineer",
    "Data scientist":                          "Data Scientist",
    "AI/ML engineer":                          "Machine Learning Engineer",
    "Applied scientist":                       "Machine Learning Engineer",
    "Architect, software or solutions":        "Full Stack Developer",
    "Cloud infrastructure engineer":           "Cloud Engineer",
    "DevOps engineer or professional":         "DevOps Engineer",
    "System administrator":                    "System Administrator",
    "Cybersecurity or InfoSec professional":   "Cybersecurity Analyst",
    "Database administrator or engineer":      "Database Administrator (DBA)",
    "UX, Research Ops or UI design professional": "UX Designer",
    "Product manager":                         "Product Manager (Teknologi)",
    "Developer, QA or test":                   "QA Engineer",
    "Support engineer or analyst":             "IT Support Specialist",
}

In [5]:
# DevType yang tidak relevan → di-drop
SO_DEVTYPE_IGNORE = {
    "Engineering manager", "Academic researcher", "Financial analyst or engineer",
    "Founder, technology or otherwise", "Other (please specify):",
    "Project manager", "Retired", "Senior executive (C-suite, VP, etc.)", "Student"
}
 

In [ ]:
# HELPER FUNCTIONS

 
def map_job_title(title: str) -> str:
    """Map job title dari Dataset 1 ke label karir target."""
    if not isinstance(title, str):
        return None
    title_lower = title.lower()
    for keyword, career in TITLE_TO_CAREER.items():
        if keyword in title_lower:
            return career
    return None
 
 
def parse_skill_list(skill_str: str) -> list:
    """Parse string representasi list Python menjadi list skill."""
    if not isinstance(skill_str, str):
        return []
    try:
        parsed = ast.literal_eval(skill_str)
        if isinstance(parsed, list):
            return [str(s).strip().lower() for s in parsed if s]
    except Exception:
        # Fallback: coba ekstrak dengan regex
        items = re.findall(r"'([^']+)'|\"([^\"]+)\"", skill_str)
        return [i[0] or i[1] for i in items]
    return []
 
 
def clean_text_list(items: list) -> list:
    """Bersihkan dan normalisasi list teks."""
    cleaned = []
    for item in items:
        if not isinstance(item, str):
            continue
        item = item.strip().lower()
        item = re.sub(r'[^\w\s\+\#\.]', '', item)  # hapus karakter aneh
        item = re.sub(r'\s+', ' ', item)            # normalisasi spasi
        if len(item) > 1:
            cleaned.append(item)
    return list(set(cleaned))  # hapus duplikat
 
 
def split_semicolon(val) -> list:
    """Split nilai multi-label yang dipisah semicolon."""
    if not isinstance(val, str):
        return []
    return [v.strip() for v in val.split(';') if v.strip()]
 
 
def encode_education(val: str) -> int:
    """Encode tingkat pendidikan ke nilai ordinal."""
    if not isinstance(val, str):
        return 1  # default: some college
    val_lower = val.lower()
    if 'phd' in val_lower or 'doctoral' in val_lower:
        return 4
    elif "master" in val_lower:
        return 3
    elif "bachelor" in val_lower:
        return 2
    elif "associate" in val_lower:
        return 1
    elif "secondary" in val_lower or "high school" in val_lower:
        return 0
    return 1
 
 
def encode_years_code(val) -> float:
    """Konversi YearsCode ke nilai numerik."""
    if pd.isna(val):
        return 2.0
    val_str = str(val).strip().lower()
    if 'less than 1' in val_str:
        return 0.5
    elif 'more than 50' in val_str:
        return 51.0
    try:
        return float(val)
    except ValueError:
        return 2.0

In [ ]:
# STEP 1: LOAD DATASET

print("=" * 60)
print("STEP 1: LOADING DATASET")
print("=" * 60)
 
# Dataset 1 — Job Postings (Parquet)
print("\n[1/2] Membaca Dataset 1: Job Postings IT (Parquet)...")
df_jobs = pd.read_parquet(PATH_PARQUET, engine='fastparquet')
print(f"      Total rows      : {len(df_jobs):,}")
print(f"      Columns         : {list(df_jobs.columns)}")
print(f"      Categories      : {df_jobs['category'].value_counts().to_dict()}")
 
# Dataset 2 — StackOverflow Survey (CSV)
print("\n[2/2] Membaca Dataset 2: StackOverflow Survey (CSV)...")
SO_COLS = [
    'DevType', 'LanguageHaveWorkedWith', 'WebframeHaveWorkedWith',
    'DatabaseHaveWorkedWith', 'DevEnvsHaveWorkedWith',
    'PlatformHaveWorkedWith', 'YearsCode', 'EdLevel'
]
df_so = pd.read_csv(PATH_CSV, usecols=SO_COLS)
print(f"      Total rows      : {len(df_so):,}")
print(f"      Null per kolom  :")
for col in SO_COLS:
    null_pct = df_so[col].isnull().mean() * 100
    print(f"        {col:<35} : {null_pct:.1f}% null")

STEP 1: LOADING DATASET

[1/2] Membaca Dataset 1: Job Postings IT (Parquet)...
      Total rows      : 1,167
      Columns         : ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set']
      Categories      : {'INFORMATION-TECHNOLOGY': 240, 'BUSINESS-DEVELOPMENT': 239, 'FINANCE': 236, 'SALES': 232, 'HR': 220}

[2/2] Membaca Dataset 2: StackOverflow Survey (CSV)...
      Total rows      : 49,191
      Null per kolom  :
        DevType                             : 11.2% null
        LanguageHaveWorkedWith              : 35.6% null
        WebframeHaveWorkedWith              : 53.3% null
        DatabaseHaveWorkedWith              : 48.1% null
        DevEnvsHaveWorkedWith               : 47.1% null
        PlatformHaveWorkedWith              : 50.7% null
        YearsCode                           : 12.5% null
        EdLevel                             : 2.1% null


In [ ]:
# STEP 2: CLEANING DATASET 1 (Job Postings IT)

print("\n" + "=" * 60)
print("STEP 2: CLEANING DATASET 1 — JOB POSTINGS IT")
print("=" * 60)
 
# 2.1 Filter kategori IT
df_it = df_jobs[df_jobs['category'] == 'INFORMATION-TECHNOLOGY'].copy()
print(f"\n[2.1] Filter IT category  : {len(df_it)} rows")
 
# 2.2 Drop duplikat
before = len(df_it)
df_it = df_it.drop_duplicates(subset=['job_id'])
print(f"[2.2] Drop duplikat       : {before} → {len(df_it)} rows")
 
# 2.3 Drop rows dengan missing values kritis
df_it = df_it.dropna(subset=['job_title', 'job_skill_set'])
print(f"[2.3] Drop null kritis    : {len(df_it)} rows")
 
# 2.4 Parse job_skill_set dari string ke list
df_it['skills_raw'] = df_it['job_skill_set'].apply(parse_skill_list)
valid_skills = df_it['skills_raw'].apply(lambda x: len(x) > 0)
df_it = df_it[valid_skills].copy()
print(f"[2.4] Parse skill list    : {len(df_it)} rows valid")
 
# 2.5 Bersihkan skill list
df_it['skills_clean'] = df_it['skills_raw'].apply(clean_text_list)
 
# 2.6 Mapping job_title ke label karir target
df_it['career_label'] = df_it['job_title'].apply(map_job_title)
before = len(df_it)
df_it = df_it.dropna(subset=['career_label'])
print(f"[2.5] Mapping career label: {before} → {len(df_it)} rows berhasil di-map")
print(f"      Label karir unik  : {sorted(df_it['career_label'].unique())}")
print(f"\n      Distribusi per label:")
for career, count in df_it['career_label'].value_counts().items():
    bar = "█" * min(count, 30)
    print(f"        {career:<40} : {count:>3}  {bar}")
 
# 2.7 Buat DataFrame bersih Dataset 1
df1_clean = pd.DataFrame({
    'source'          : 'job_postings',
    'skills'          : df_it['skills_clean'].apply(lambda x: ';'.join(x)),
    'tools'           : '',        # Dataset 1 tidak memiliki kolom tools terpisah
    'languages'       : '',        # Dataset 1 tidak memiliki kolom bahasa pemrograman terpisah
    'databases'       : '',
    'years_code'      : np.nan,
    'education_level' : np.nan,
    'career_label'    : df_it['career_label'].values
})
 
print(f"\n[2.6] Dataset 1 clean shape: {df1_clean.shape}")


STEP 2: CLEANING DATASET 1 — JOB POSTINGS IT

[2.1] Filter IT category  : 240 rows
[2.2] Drop duplikat       : 240 → 240 rows
[2.3] Drop null kritis    : 240 rows
[2.4] Parse skill list    : 240 rows valid
[2.5] Mapping career label: 240 → 59 rows berhasil di-map
      Label karir unik  : ['Business Intelligence Analyst', 'Cybersecurity Analyst', 'Data Analyst', 'Database Administrator (DBA)', 'DevOps Engineer', 'Helpdesk Analyst', 'IT Support Specialist', 'Network Engineer', 'Product Manager (Teknologi)', 'System Administrator']

      Distribusi per label:
        IT Support Specialist                    :  26  ██████████████████████████
        Cybersecurity Analyst                    :   8  ████████
        System Administrator                     :   8  ████████
        Helpdesk Analyst                         :   7  ███████
        Network Engineer                         :   3  ███
        Business Intelligence Analyst            :   3  ███
        Database Administrator (DBA) 

In [ ]:
# STEP 3: CLEANING DATASET 2 (StackOverflow Survey)
print("\n" + "=" * 60)
print("STEP 3: CLEANING DATASET 2 — STACKOVERFLOW SURVEY")
print("=" * 60)
 
# 3.1 Drop baris tanpa DevType (tidak punya label karir)
before = len(df_so)
df_so = df_so.dropna(subset=['DevType']).copy()
print(f"\n[3.1] Drop null DevType   : {before} → {len(df_so)} rows")
 
# 3.2 Explode DevType (bisa multi-value, ambil nilai pertama sebagai label utama)
#     Note: satu responden bisa punya beberapa DevType, ambil yang pertama
df_so['DevType_primary'] = df_so['DevType'].apply(
    lambda x: split_semicolon(x)[0] if split_semicolon(x) else None
)
 
# 3.3 Filter hanya DevType yang relevan ke 35 karir target
df_so['career_label'] = df_so['DevType_primary'].map(SO_DEVTYPE_TO_CAREER)
before = len(df_so)
df_so = df_so.dropna(subset=['career_label']).copy()
print(f"[3.2] Mapping DevType     : {before} → {len(df_so)} rows relevan")
print(f"      Label karir unik  : {df_so['career_label'].nunique()} karir")
print(f"\n      Distribusi per label:")
for career, count in df_so['career_label'].value_counts().items():
    bar = "█" * min(count // 20, 30)
    print(f"        {career:<40} : {count:>5}  {bar}")
 
# 3.4 Encode YearsCode ke numerik
df_so['years_code'] = df_so['YearsCode'].apply(encode_years_code)
 
# 3.5 Encode EdLevel ke ordinal
df_so['education_level'] = df_so['EdLevel'].apply(encode_education)
 
# 3.6 Gabungkan semua kolom tools/languages/databases/devenvs ke dalam field masing-masing
df_so['languages_list'] = df_so['LanguageHaveWorkedWith'].apply(split_semicolon)
df_so['tools_list']     = df_so['WebframeHaveWorkedWith'].apply(split_semicolon)
df_so['databases_list'] = df_so['DatabaseHaveWorkedWith'].apply(split_semicolon)
df_so['devenvs_list']   = df_so['DevEnvsHaveWorkedWith'].apply(split_semicolon)
 
# Gabungkan tools + devenvs sebagai "tools" untuk keseragaman
df_so['tools_combined'] = df_so.apply(
    lambda r: list(set(r['tools_list'] + r['devenvs_list'])), axis=1
)
 
# Lowercase & clean
df_so['languages_clean'] = df_so['languages_list'].apply(
    lambda lst: [x.strip().lower() for x in lst if x.strip()]
)
df_so['tools_clean'] = df_so['tools_combined'].apply(
    lambda lst: [x.strip().lower() for x in lst if x.strip()]
)
df_so['databases_clean'] = df_so['databases_list'].apply(
    lambda lst: [x.strip().lower() for x in lst if x.strip()]
)
 
# 3.7 Buat DataFrame bersih Dataset 2
df2_clean = pd.DataFrame({
    'source'          : 'so_survey',
    'skills'          : '',   # SO tidak punya kolom skills eksplisit
    'tools'           : df_so['tools_clean'].apply(lambda x: ';'.join(x)),
    'languages'       : df_so['languages_clean'].apply(lambda x: ';'.join(x)),
    'databases'       : df_so['databases_clean'].apply(lambda x: ';'.join(x)),
    'years_code'      : df_so['years_code'].values,
    'education_level' : df_so['education_level'].values,
    'career_label'    : df_so['career_label'].values
})
 
print(f"\n[3.3] Dataset 2 clean shape: {df2_clean.shape}")


STEP 3: CLEANING DATASET 2 — STACKOVERFLOW SURVEY

[3.1] Drop null DevType   : 49191 → 43680 rows
[3.2] Mapping DevType     : 43680 → 34835 rows relevan
      Label karir unik  : 18 karir

      Distribusi per label:
        Full Stack Developer                     : 15035  ██████████████████████████████
        Backend Developer                        :  9646  ██████████████████████████████
        Frontend Developer                       :  1974  ██████████████████████████████
        Mobile App Developer                     :  1391  ██████████████████████████████
        Machine Learning Engineer                :  1210  ██████████████████████████████
        DevOps Engineer                          :  1053  ██████████████████████████████
        Data Engineer                            :   770  ██████████████████████████████
        Data Scientist                           :   574  ████████████████████████████
        System Administrator                     :   480  ██████████████

In [ ]:
# STEP 4: PENGGABUNGAN KEDUA DATASET

print("\n" + "=" * 60)
print("STEP 4: PENGGABUNGAN DATASET")
print("=" * 60)
 
df_merged = pd.concat([df1_clean, df2_clean], ignore_index=True)
print(f"\n[4.1] Shape setelah concat : {df_merged.shape}")
print(f"      Sumber data:")
print(f"        job_postings : {len(df1_clean)} rows")
print(f"        so_survey    : {len(df2_clean)} rows")
print(f"        Total        : {len(df_merged)} rows")
 


STEP 4: PENGGABUNGAN DATASET

[4.1] Shape setelah concat : (34894, 8)
      Sumber data:
        job_postings : 59 rows
        so_survey    : 34835 rows
        Total        : 34894 rows


In [ ]:
# STEP 5: POST-MERGE CLEANING

print("\n" + "=" * 60)
print("STEP 5: POST-MERGE CLEANING")
print("=" * 60)
 
# 5.1 Gabungkan skills + languages sebagai satu kolom "all_skills"
#     (karena Dataset 1 punya skills, Dataset 2 punya languages — keduanya = skill teknis)
def merge_skill_fields(row):
    combined = set()
    if isinstance(row['skills'], str) and row['skills']:
        combined.update([s.strip() for s in row['skills'].split(';') if s.strip()])
    if isinstance(row['languages'], str) and row['languages']:
        combined.update([s.strip() for s in row['languages'].split(';') if s.strip()])
    return ';'.join(sorted(combined))
 
df_merged['all_skills'] = df_merged.apply(merge_skill_fields, axis=1)
print(f"\n[5.1] Kolom 'all_skills' dibuat (gabungan skills + languages)")
 
# 5.2 Bersihkan tools & databases
df_merged['tools']     = df_merged['tools'].fillna('')
df_merged['databases'] = df_merged['databases'].fillna('')
 
# 5.3 Imputasi missing values numerik
median_years = df_merged['years_code'].median()
median_edu   = df_merged['education_level'].median()
df_merged['years_code']      = df_merged['years_code'].fillna(median_years)
df_merged['education_level'] = df_merged['education_level'].fillna(median_edu).astype(int)
print(f"[5.2] Imputasi years_code  : median = {median_years:.1f} tahun")
print(f"[5.3] Imputasi edu_level   : median = {int(median_edu)} (2=Bachelor)")
 
# 5.4 Drop rows dimana all_skills DAN tools keduanya kosong
mask_empty = (df_merged['all_skills'] == '') & (df_merged['tools'] == '')
before = len(df_merged)
df_merged = df_merged[~mask_empty].copy()
print(f"[5.4] Drop rows tanpa skill & tools: {before} → {len(df_merged)} rows")
 
# 5.5 Drop duplikat berdasarkan kombinasi all_skills + tools + career_label
before = len(df_merged)
df_merged = df_merged.drop_duplicates(
    subset=['all_skills', 'tools', 'career_label']
).copy()
print(f"[5.5] Drop duplikat        : {before} → {len(df_merged)} rows")
 
# 5.6 Reset index
df_merged = df_merged.reset_index(drop=True)


STEP 5: POST-MERGE CLEANING

[5.1] Kolom 'all_skills' dibuat (gabungan skills + languages)
[5.2] Imputasi years_code  : median = 14.0 tahun
[5.3] Imputasi edu_level   : median = 2 (2=Bachelor)
[5.4] Drop rows tanpa skill & tools: 34894 → 25893 rows
[5.5] Drop duplikat        : 25893 → 25038 rows


In [ ]:
# STEP 6: VALIDASI & SUMMARY

print("\n" + "=" * 60)
print("STEP 6: VALIDASI AKHIR & SUMMARY")
print("=" * 60)
 
print(f"\n[6.1] Shape final          : {df_merged.shape}")
print(f"[6.2] Kolom                : {list(df_merged.columns)}")
print(f"\n[6.3] Missing values final :")
print(df_merged.isnull().sum().to_string())
 
print(f"\n[6.4] Distribusi label karir FINAL:")
label_dist = df_merged['career_label'].value_counts()
for career, count in label_dist.items():
    bar = "█" * min(count // 5, 40)
    pct = count / len(df_merged) * 100
    print(f"  {career:<42}: {count:>5} ({pct:4.1f}%)  {bar}")
 
print(f"\n[6.5] Total karir unik     : {df_merged['career_label'].nunique()}")
print(f"[6.6] Total rows final     : {len(df_merged):,}")
 
# Cek class imbalance
min_class = label_dist.min()
max_class = label_dist.max()
ratio = max_class / min_class
print(f"\n[6.7] Class imbalance ratio: {ratio:.1f}x")
if ratio > 10:
    print("      ⚠️  PERLU SMOTE — imbalance > 10x")
elif ratio > 3:
    print("      ⚠️  Sebaiknya terapkan SMOTE atau class_weight='balanced'")
else:
    print("      ✅ Distribusi cukup seimbang")
 
# Sample output
print("\n[6.8] Sample 5 baris pertama dataset final:")
print(df_merged[['source','all_skills','tools','databases',
                  'years_code','education_level','career_label']].head(5).to_string())


STEP 6: VALIDASI AKHIR & SUMMARY

[6.1] Shape final          : (25038, 9)
[6.2] Kolom                : ['source', 'skills', 'tools', 'languages', 'databases', 'years_code', 'education_level', 'career_label', 'all_skills']

[6.3] Missing values final :
source             0
skills             0
tools              0
languages          0
databases          0
years_code         0
education_level    0
career_label       0
all_skills         0

[6.4] Distribusi label karir FINAL:
  Full Stack Developer                      : 11062 (44.2%)  ████████████████████████████████████████
  Backend Developer                         :  6886 (27.5%)  ████████████████████████████████████████
  Frontend Developer                        :  1352 ( 5.4%)  ████████████████████████████████████████
  Mobile App Developer                      :   870 ( 3.5%)  ████████████████████████████████████████
  Machine Learning Engineer                 :   810 ( 3.2%)  ████████████████████████████████████████
  DevOps En

In [ ]:
# STEP 7: SIMPAN OUTPUT

print("\n" + "=" * 60)
print("STEP 7: MENYIMPAN DATASET")
print("=" * 60)
 
# Kolom final yang disimpan
FINAL_COLS = [
    'source', 'all_skills', 'tools', 'databases',
    'years_code', 'education_level', 'career_label'
]
df_final = df_merged[FINAL_COLS].copy()
df_final.to_csv(PATH_OUTPUT, index=False, encoding='utf-8')
print(f"\n✅ Dataset berhasil disimpan: '{PATH_OUTPUT}'")
print(f"   Shape      : {df_final.shape}")
print(f"   Size       : {os.path.getsize(PATH_OUTPUT) / 1024:.1f} KB")
 
print("\n" + "=" * 60)
print("PIPELINE SELESAI ✅")
print("=" * 60)
print("""
LANGKAH SELANJUTNYA:
──────────────────────────────────────────────────────────────
1. EDA        : Jalankan notebook EDA pada dataset_it_careers_clean.csv
2. Encoding   : Gunakan MultiLabelBinarizer untuk kolom all_skills, tools, databases
3. SMOTE      : Terapkan jika class imbalance > 3x
4. Training   : Bandingkan Random Forest, SVM, KNN dengan cross-validation
5. Deployment : Simpan model + encoder ke .pkl → integrasikan ke Streamlit
──────────────────────────────────────────────────────────────
""")


STEP 7: MENYIMPAN DATASET

✅ Dataset berhasil disimpan: 'dataset_it_careers_clean.csv'
   Shape      : (25038, 7)
   Size       : 4378.0 KB

PIPELINE SELESAI ✅

LANGKAH SELANJUTNYA:
──────────────────────────────────────────────────────────────
1. EDA        : Jalankan notebook EDA pada dataset_it_careers_clean.csv
2. Encoding   : Gunakan MultiLabelBinarizer untuk kolom all_skills, tools, databases
3. SMOTE      : Terapkan jika class imbalance > 3x
4. Training   : Bandingkan Random Forest, SVM, KNN dengan cross-validation
5. Deployment : Simpan model + encoder ke .pkl → integrasikan ke Streamlit
──────────────────────────────────────────────────────────────



In [ ]:
# Step 8 Load dataset hasil cleaning yang sudah disimpan
df = pd.read_csv("dataset_it_careers_clean.csv")



Total baris saat ini: 25038
Baris dengan 'tools' kosong: 1836
Baris dengan 'databases' kosong: 4174
-------------------------------------------
Total baris yang akan hilang jika dihapus: 5186
Sisa data jika tetap dihapus: 19852


In [ ]:
# step 9 Analisis missing value
# Definisikan apa yang dianggap "Null" (String Kosong)
empty_tools = df[df['tools'].isna() | (df['tools'] == '')]
empty_databases = df[df['databases'].isna() | (df['databases'] == '')]

# Hitung total baris yang akan terhapus jika kita buang yang kosong
# Baris yang kosong di 'tools' ATAU 'databases'
rows_to_drop = df[(df['tools'].isna() | (df['tools'] == '')) | 
                  (df['databases'].isna() | (df['databases'] == ''))]

print(f"Total baris saat ini: {len(df)}")
print(f"Baris dengan 'tools' kosong: {len(empty_tools)}")
print(f"Baris dengan 'databases' kosong: {len(empty_databases)}")
print(f"-------------------------------------------")
print(f"Total baris yang akan hilang jika dihapus: {len(rows_to_drop)}")
print(f"Sisa data jika tetap dihapus: {len(df) - len(rows_to_drop)}")

Total baris saat ini: 25038
Baris dengan 'tools' kosong: 1836
Baris dengan 'databases' kosong: 4174
-------------------------------------------
Total baris yang akan hilang jika dihapus: 5186
Sisa data jika tetap dihapus: 19852


Evaluasi Kualitas Data dan Analisis Nilai Kosong (Missing Values)
Tahap ini bertujuan untuk mengevaluasi kualitas dataset setelah proses pembersihan awal, dengan fokus pada kolom teknis utama yaitu tools dan databases. Mengingat dataset berasal dari penggabungan dua sumber yang berbeda, seringkali ditemukan inkonsistensi data dalam bentuk nilai kosong.

- Tujuan Analisis:
Identifikasi Nilai Kosong: Mendeteksi data yang bernilai NaN atau string kosong ('') yang tidak informatif bagi model.

- Analisis Dampak: Menghitung seberapa banyak data yang akan hilang jika dilakukan penghapusan baris (drop rows) secara permanen.

- Justifikasi Strategi: Menentukan apakah dataset lebih baik dibersihkan dengan cara dihapus atau diisi dengan nilai konstanta (imputasi) agar informasi tetap terjaga.

Logika Pengecekan:
Analisis menggunakan logika OR (|), di mana sebuah baris dikategorikan sebagai "berisiko hilang" jika salah satu atau kedua kolom (tools atau databases) tidak memiliki data. Hal ini penting untuk menjaga integritas fitur saat masuk ke tahap Feature Engineering (Encoding).

In [ ]:
# Step 10 Tahap preprocessing & feature engineering untuk mengubah skill pengguna menjadi fitur numerik
from sklearn.preprocessing import MultiLabelBinarizer

# Pastikan kolom skill menjadi list
df['skill_list'] = df['all_skills'].apply(lambda x: str(x).split(';') if pd.notna(x) else [])

# Inisialisasi dan Transformasi MultiLabelBinarizer
mlb = MultiLabelBinarizer()
skill_encoded = mlb.fit_transform(df['skill_list'])


# Buat DataFrame fitur baru
df_features = pd.DataFrame(skill_encoded, columns=mlb.classes_)

# Gabungkan dengan fitur numerik lainnya (years_code & education_level)
X = pd.concat([df[['years_code', 'education_level']], df_features], axis=1)
y = df['career_label']

print(f"Fitur siap! Sekarang kamu punya {X.shape[1]} kolom fitur.")

Fitur siap! Sekarang kamu punya 619 kolom fitur.


transformasi Fitur Menggunakan Multi-label Encoding

Tahap ini merupakan bagian dari Feature Engineering yang bertujuan untuk mengubah data tekstual pada kolom all_skills menjadi format numerik. Karena setiap baris data dapat memiliki lebih dari satu keahlian (multi-label), metode MultiLabelBinarizer digunakan untuk merepresentasikan kehadiran setiap skill dalam bentuk biner (0 dan 1).

Tujuan Proses:
- Vektorisasi Skill: Mengonversi daftar skill yang dipisahkan oleh titik koma (;) menjadi kolom-kolom baru (fitur) yang independen.
- Integrasi Fitur: Menggabungkan fitur hasil encoding dengan fitur numerik yang sudah ada, seperti years_code (pengalaman) dan education_level (tingkat pendidikan).
- Persiapan Input Model: Menghasilkan matriks fitur ($X$) dan vektor target ($y$) yang siap diproses oleh 

algoritma Machine Learning.Mekanisme Kerja:Setiap skill unik yang ditemukan di seluruh dataset akan menjadi satu kolom baru.Jika seorang responden memiliki skill tersebut, maka kolom akan bernilai 1, jika tidak maka bernilai 0.Proses ini sangat penting untuk memastikan data kualitatif dapat dihitung secara matematis oleh model klasifikasi.

In [ ]:
from sklearn.preprocessing import StandardScaler
# Step 12 Feature Scaling

# 1. Menghitung total skill yang dimiliki (Fitur Baru)
X['total_skills'] = df_features.sum(axis=1)

# 2. Scaling fitur numerik (years_code dan total_skills)
scaler = StandardScaler()
X[['years_code', 'total_skills']] = scaler.fit_transform(X[['years_code', 'total_skills']])

print("Feature Engineering Tambahan Selesai: Scaling dan Total Skills sudah diterapkan.")

Feature Engineering Tambahan Selesai: Scaling dan Total Skills sudah diterapkan.


In [ ]:
from imblearn.over_sampling import SMOTE
# Step 13 Data Balancing
# Menggunakan SMOTE untuk menyeimbangkan kelas
# k_neighbors diperkecil karena ada karir yang datanya sangat sedikit (misal cuma 2-3 data)
sm = SMOTE(random_state=42, k_neighbors=1) 
X_resampled, y_resampled = sm.fit_resample(X, y)

print(f"Sebelum SMOTE: {len(X)} baris")
print(f"Sesudah SMOTE: {len(X_resampled)} baris")
print("Data sekarang sudah seimbang!")


Sebelum SMOTE: 25038 baris
Sesudah SMOTE: 232302 baris
Data sekarang sudah seimbang!


In [ ]:
#step 14 Spliting Data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

In [ ]:
# step 15 Model Training
from sklearn.ensemble import RandomForestClassifier

model_karir = RandomForestClassifier(n_estimators=100, random_state=42)
model_karir.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
# Step 16 Model Evaluation
from sklearn.metrics import classification_report

y_pred = model_karir.predict(X_test)
print(classification_report(y_test, y_pred))

                               precision    recall  f1-score   support

            Backend Developer       0.65      0.66      0.65      2194
Business Intelligence Analyst       0.71      0.70      0.71      2213
               Cloud Engineer       0.96      0.95      0.95      2198
        Cybersecurity Analyst       0.98      0.96      0.97      2153
                 Data Analyst       0.95      0.95      0.95      2216
                Data Engineer       0.92      0.91      0.91      2243
               Data Scientist       0.92      0.93      0.92      2234
 Database Administrator (DBA)       0.97      0.98      0.98      2239
              DevOps Engineer       0.92      0.90      0.91      2193
           Frontend Developer       0.88      0.89      0.88      2191
         Full Stack Developer       0.62      0.63      0.62      2186
               Game Developer       0.98      0.97      0.97      2157
             Helpdesk Analyst       1.00      1.00      1.00      2163
     

In [ ]:
# Step 17 Model Saving untuk Deployment Streamlit
from sklearn.preprocessing import LabelEncoder
import joblib

# Encode label career
le = LabelEncoder()
y = le.fit_transform(df['career_label'])

# Simpan semua file
joblib.dump(model_karir, "model_karir.pkl")
joblib.dump(mlb, "mlb_skills.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")

print("✅ Semua file tersimpan dan siap untuk Streamlit!")
print(f"   - model_karir.pkl")
print(f"   - mlb_skills.pkl    → {len(mlb.classes_)} skill unik")
print(f"   - scaler.pkl")
print(f"   - label_encoder.pkl → {len(le.classes_)} label karir")

✅ Semua file tersimpan dan siap untuk Streamlit!
   - model_karir.pkl
   - mlb_skills.pkl    → 617 skill unik
   - scaler.pkl
   - label_encoder.pkl → 21 label karir
